# 12 - Focal Calibration & Geometric Height  (Track B)

The payoff of Track B: turn a bounding box + USS distance into a **metric
object height**, at zero training cost.

Pinhole model: an object of real height H at distance Z has pixel height
`h = f*H/Z`. So calibrate the camera focal length `f` (one constant) from
objects whose real height is known, then invert for any detection:

```
f       = median( h_px * Z / H )   over dummychild (125 cm) + pole (108 cm)
height  = h_px * Z / f
```

**Inputs** (both already produced):
- `frame_distances.csv` from notebook 11 - per (sequence, frame) USS distance,
  ACCEPTED sequences only.
- The label manifest / boxes - per (sequence, frame) box pixel height.

**Validation** (the honest part):
- fit `f` on part of the anchors, report height error on the HELD-OUT part;
- Low/High accuracy from the geometric height vs the 25 cm threshold, compared
  to the class-derived Low/High - does geometry alone recover the operational
  decision?

This is a *feasibility* result: monocular height is coarse (expect tens of %),
but a coarse height is enough for the <25 cm drive-over call, and it is free.

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    assert (REPO_DIR / ".git").exists(), \
        "repo not found - run scripts/bootstrap_colab.sh first (set REPO_URL)"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None
RAW_ROOT_OVERRIDE = None

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")

In [ ]:
import json as _json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from vision_uss_research.alignment.height import (KNOWN_HEIGHT_MM,
                                                 LOW_HIGH_THRESHOLD_MM,
                                                 estimate_focal, estimate_height_mm,
                                                 height_error, low_high_from_height)
from vision_uss_research.csv_io import read_csv_tolerant
from vision_uss_research.labeling.boxes_io import load_boxes_tolerant
from vision_uss_research.runs import finish_run, new_run_id, resolve_run, start_run
from vision_uss_research.sequences import frames_window_id

SEED = 0
CALIB_FRAC = 0.7          # anchor sequences used to FIT focal; rest held out
MIN_BOX_PX = 8            # ignore degenerate boxes
DISTANCE_RUN = None       # alignment/<run_id> with frame_distances.csv; None=newest
LABELS_SOURCE = "auto"    # or explicit list of boxes CSVs

# distance run (notebook 11 output)
dist_run = (ARTIFACTS / "alignment" / DISTANCE_RUN) if DISTANCE_RUN else resolve_run(
    ARTIFACTS / "alignment", "alignment", require_file="frame_distances.csv")
assert dist_run is not None, "run notebook 11 first (frame_distances.csv)"
SEQ_QUALITY = dist_run / "sequence_distance_quality.csv"

RUN_DIR = ARTIFACTS / "alignment" / new_run_id("alignment", "height")
RUN_DIR.mkdir(parents=True, exist_ok=True)
start_run(RUN_DIR, config={"calib_frac": CALIB_FRAC, "seed": SEED,
                           "threshold_mm": LOW_HIGH_THRESHOLD_MM,
                           "known_heights": KNOWN_HEIGHT_MM},
          inputs=[f"alignment/{dist_run.name}"], repo_root=REPO_DIR)
print(f"distance run : {dist_run.name}")
print(f"output       : {RUN_DIR}")

## Join per-frame distance to box pixel heights

In [ ]:
# per-frame distances, ACCEPTED sequences only
fd = pd.read_csv(dist_run / "frame_distances.csv")
fd["distance_mm"] = pd.to_numeric(fd["distance_mm"], errors="coerce")
if SEQ_QUALITY.exists():
    q = read_csv_tolerant(SEQ_QUALITY, list(pd.read_csv(SEQ_QUALITY, nrows=0).columns))
    accepted = set(q[pd.to_numeric(q["accepted"], errors="coerce") == 1]
                   ["sequence_id"].astype(str))
    fd = fd[fd["sequence_id"].astype(str).isin(accepted)]
print(f"distance rows (accepted seqs): {len(fd)}, "
      f"{fd['sequence_id'].nunique()} sequences")

# label boxes: auto-discover the same runs notebook 05 merges
WINDOW_ROOT = ARTIFACTS / "frames" / frames_window_id(20, (0.30, 0.85))
if LABELS_SOURCE == "auto":
    boxes_csvs = []
    for base in [ARTIFACTS / "labels" / "runs", ARTIFACTS / "labels" / "clicks"]:
        for run_dir in sorted(base.glob("*")) if base.exists() else []:
            if (run_dir / "propagated_boxes.csv").exists():
                boxes_csvs.append(run_dir / "propagated_boxes.csv")
else:
    boxes_csvs = [Path(p) for p in LABELS_SOURCE]
assert boxes_csvs, "no label runs found (notebooks 02/03)"

boxes = pd.concat([load_boxes_tolerant(p) for p in boxes_csvs], ignore_index=True)
boxes = boxes[boxes["x0"].notna()].copy()
boxes["box_px_height"] = (boxes["y1"] - boxes["y0"]).abs()
# one box per (sequence, frame): the target is the largest box in frame
boxes = (boxes.sort_values("box_px_height", ascending=False)
         .drop_duplicates(["sequence_id", "camera", "frame_idx"], keep="first"))

joined = fd.merge(
    boxes[["sequence_id", "camera", "frame_idx", "target_object",
           "box_px_height"]],
    on=["sequence_id", "camera", "frame_idx"], how="inner")
joined = joined[(joined["box_px_height"] >= MIN_BOX_PX)
                & joined["distance_mm"].notna()]
print(f"joined (frame has BOTH distance and a box): {len(joined)} rows, "
      f"{joined['sequence_id'].nunique()} sequences")
display(joined.groupby("target_object").size().sort_values(ascending=False)
        .head(15).to_frame("frames_with_distance_and_box"))

## Calibrate focal on held-out anchors

In [ ]:
anchors = joined[joined["target_object"].isin(KNOWN_HEIGHT_MM)].copy()
anchors["real_height_mm"] = anchors["target_object"].map(KNOWN_HEIGHT_MM)
print(f"anchor frames: {len(anchors)} "
      f"({anchors.groupby('target_object').size().to_dict()})")

# split anchors by SEQUENCE (no frame leakage) into calib / holdout
rng = np.random.default_rng(SEED)
anchor_seqs = sorted(anchors["sequence_id"].astype(str).unique())
rng.shuffle(anchor_seqs)
n_cal = max(1, int(CALIB_FRAC * len(anchor_seqs)))
cal_seqs, hold_seqs = set(anchor_seqs[:n_cal]), set(anchor_seqs[n_cal:])
cal = anchors[anchors["sequence_id"].astype(str).isin(cal_seqs)]
hold = anchors[anchors["sequence_id"].astype(str).isin(hold_seqs)]

focal = estimate_focal(cal["box_px_height"], cal["distance_mm"],
                       cal["real_height_mm"])
print(f"\ncalibrated focal length: {focal:.1f} px "
      f"(from {len(cal)} frames / {len(cal_seqs)} sequences)")

# validate on held-out anchor frames
hold = hold.assign(pred_mm=estimate_height_mm(hold["box_px_height"],
                                              hold["distance_mm"], focal))
err = height_error(hold["pred_mm"], hold["real_height_mm"])
print(f"held-out height error ({err['n']} frames / {len(hold_seqs)} seqs):")
print(f"  MAE  : {err['mae_mm']:.0f} mm")
print(f"  RMSE : {err['rmse_mm']:.0f} mm")
print(f"  mean relative error: {100*err['mean_rel_err']:.1f}%")

In [ ]:
# predicted vs true height on held-out anchors
if len(hold):
    fig, ax = plt.subplots(figsize=(5, 5))
    for obj, g in hold.groupby("target_object"):
        ax.scatter(g["real_height_mm"], g["pred_mm"], s=12, alpha=0.5, label=obj)
    lim = [0, max(hold["real_height_mm"].max(), hold["pred_mm"].max()) * 1.1]
    ax.plot(lim, lim, "k--", lw=1, label="ideal")
    ax.set_xlabel("true height (mm)"); ax.set_ylabel("predicted height (mm)")
    ax.set_title("Geometric height vs known height (held-out)")
    ax.legend(); ax.set_xlim(lim); ax.set_ylim(lim)
    fig.savefig(RUN_DIR / "height_calibration.jpg", dpi=110, bbox_inches="tight")
    plt.show()

## Low/High from geometry vs the class-derived label

The operational question: does the geometric height alone recover the <25 cm
drive-over decision? Compare Low/High derived from the estimated height against
Low/High from the object class, on ALL joined detections (not just anchors).

In [ ]:
# class-derived Low/High (same map as notebooks 07-09)
CLASS_BINS = {"curbstone": "low", "speedbump": "low", "woodenboard": "low",
              "pole": "high", "dummychild": "high", "bicyclestand": "high",
              "hose": "low", "step": "low", "cone": "high", "bollard": "high",
              "car": "high", "tree": "high", "bush": "high", "stonelarge": "high",
              "stonemiddle": "high", "cubestandard": "high", "standardcube": "high",
              "ubarrier": "high", "fence": "high", "curbstone_side": "low"}

ev = joined.copy()
ev["height_mm"] = estimate_height_mm(ev["box_px_height"], ev["distance_mm"], focal)
ev["geo_bin"] = low_high_from_height(ev["height_mm"])
ev["class_bin"] = ev["target_object"].str.lower().map(CLASS_BINS)
ev = ev[ev["geo_bin"].notna() & ev["class_bin"].notna()]

# aggregate to ONE decision per sequence (median height) - the deployment unit
seq = (ev.groupby(["sequence_id", "target_object", "class_bin"])
       .agg(median_height_mm=("height_mm", "median")).reset_index())
seq["geo_bin"] = low_high_from_height(seq["median_height_mm"])
acc = (seq["geo_bin"] == seq["class_bin"]).mean()
print(f"per-sequence Low/High agreement (geometry vs class): {100*acc:.1f}% "
      f"of {len(seq)} sequences")

conf = pd.crosstab(seq["class_bin"], seq["geo_bin"],
                   rownames=["class"], colnames=["geometry"])
display(conf)
seq.to_csv(RUN_DIR / "sequence_height_lowhigh.csv", index=False)

# per-class median height - does it order sensibly by real height?
byobj = (seq.groupby("target_object")
         .agg(n=("sequence_id", "nunique"),
              median_height_mm=("median_height_mm", "median"),
              geo_low_frac=("geo_bin", lambda s: (s == "low").mean()))
         .sort_values("median_height_mm"))
display(byobj.round(1))

In [ ]:
summary = {
    "focal_px": round(float(focal), 1),
    "heldout_mae_mm": round(err["mae_mm"], 1) if err["mae_mm"] else None,
    "heldout_rel_err": round(err["mean_rel_err"], 3) if err["mean_rel_err"] else None,
    "lowhigh_agreement": round(float(acc), 4),
    "sequences_evaluated": int(len(seq)),
    "anchor_frames": int(len(anchors)),
}
finish_run(RUN_DIR, summary=summary)
print(_json.dumps(summary, indent=1))
print(f"\nper-sequence height + Low/High : {RUN_DIR / 'sequence_height_lowhigh.csv'}")

## Reading the result

- **Held-out MAE / relative error** is the headline height accuracy. Tens of %
  is expected and honest for monocular height; the USS distance is what makes
  it possible at all (a camera alone cannot).
- **Low/High agreement** is the operational number: if geometry alone recovers
  the drive-over decision at high accuracy, that is the practical payoff, and
  it complements the detector's 99.8% class-based Low/High as an independent
  signal.
- **Per-class median height** should order sensibly (curbs lowest, poles/
  dummies highest); if it does, the calibration is physically sound.

Caveats to state in the paper: single camera/vehicle; box height includes the
object's ground contact, so tall thin objects and ground-plane pitch add error;
distance is the nearest-object USS track, best on the accepted (clean-approach)
subset. Next: fuse this geometric Low/High with the USS ClassProbHigh and the
detector output (the three-way fusion experiment).